# DS 542 Fall 2025 Project 3

Your task for this project is to train an attention-based decoder-only model for math expressions with positive integers, addition, and parenthesis.
A sample model is provided and demonstrated on small problems with single digit integer inputs.
Your goal is to scale up this model to handle two digit inputs and longer expressions.

## Problem Setup

In [35]:
import math
import random

import torch

In [36]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [37]:
characters = "()+0123456789="
TOKENS = ["<bos>", "<eos>", "<pad>"] + [c for c in characters]
print(TOKENS)

['<bos>', '<eos>', '<pad>', '(', ')', '+', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '=']


In [38]:
TOKEN_MAP = dict((t, i) for i, t in enumerate(TOKENS))
print(TOKEN_MAP)

{'<bos>': 0, '<eos>': 1, '<pad>': 2, '(': 3, ')': 4, '+': 5, '0': 6, '1': 7, '2': 8, '3': 9, '4': 10, '5': 11, '6': 12, '7': 13, '8': 14, '9': 15, '=': 16}


In [39]:
BOS = TOKEN_MAP["<bos>"]
EOS = TOKEN_MAP["<eos>"]
PAD = TOKEN_MAP["<pad>"]

In [40]:
def decode(token_ids):
    return "".join(TOKENS[i] for i in token_ids)

decode([0, 3, 7, 5, 6, 4, 1, 2])

'<bos>(1+0)<eos><pad>'

In [7]:
def encode(s, *, eos=True):
    if s.startswith("<bos>"):
        s = s[5:]

    output = [BOS]
    output.extend(TOKEN_MAP[c] for c in s)

    if eos:
        output.append(EOS)

    return torch.tensor(output, device=device)

decode(encode("1+2=3"))

'<bos>1+2=3<eos>'

### Problem Generation

This function `generate_instance` will generate a random expression starting from `n` random integers between `value_min` and `value_max` (inclusive) and combining them with addition in a random order.
The full expression consists of multiple rounds of reductions of the innermost parentheses replacing the parenthesized addition with its integer value.
The final value after the last equals sign is the value of the original expression before the first equals sign.

Here are some example expressions.

* `(3+4)+(9+2)=(7+11)=18`
* `(((((1+2)+3)+4)+5)+6)=((((3+3)+4)+5)+6)=(((6+4)+5)+6)=((10+5)+6)=(15+6)=21`

To be clear, each reduction step should replace all the parenthesis that only contain two numbers being added.


In [8]:
# DO NOT CHANGE

def generate_instance(n, *, value_min=1, value_max=9):
    current_numbers = [random.randint(value_min, value_max) for _ in range(n)]
    current_expressions = [[str(v) for v in current_numbers]]
    current_fresh = [True for _ in current_numbers]

    while len(current_numbers) > 1:
        next_numbers = []
        next_expressions = [[] for _ in range(len(current_expressions) + 1)]
        next_fresh = []

        i = 0
        while i < len(current_numbers):
            can_merge = (i + 1 < len(current_numbers)) and (current_fresh[i] or current_fresh[i + 1])
            if can_merge and random.random() < 0.5:
                # decided to merge
                next_numbers.append(current_numbers[i] + current_numbers[i + 1])

                next_expressions[0].append(str(next_numbers[-1]))
                for j in range(len(current_expressions)):
                    next_expressions[j + 1].append(f"({current_expressions[j][i]}+{current_expressions[j][i + 1]})")

                next_fresh.append(True)
                i += 2
            else:
                # decided not to merge
                next_numbers.append(current_numbers[i])

                next_expressions[0].append(str(next_numbers[-1]))
                for j in range(len(current_expressions)):
                    next_expressions[j + 1].append(current_expressions[j][i])

                next_fresh.append(False)
                i += 1

        if len(next_numbers) < len(current_numbers):
            current_numbers = next_numbers
            current_expressions = next_expressions
            current_fresh = next_fresh

    output = '='.join(e[0] for e in reversed(current_expressions))
    return encode(output)

decode(generate_instance(3))

'<bos>(8+(8+4))=(8+12)=20<eos>'

In [9]:
generate_instance(3)

tensor([ 0,  3,  3, 13,  5,  9,  4,  5, 10,  4, 16,  3,  7,  6,  5, 10,  4, 16,
         7, 10,  1], device='cuda:0')

In [10]:
for i in range(10):
    print(decode(generate_instance(5)))

<bos>(((8+7)+(2+7))+9)=((15+9)+9)=(24+9)=33<eos>
<bos>(((6+4)+9)+(4+6))=((10+9)+10)=(19+10)=29<eos>
<bos>((((1+3)+5)+8)+9)=(((4+5)+8)+9)=((9+8)+9)=(17+9)=26<eos>
<bos>((1+4)+((2+2)+9))=(5+(4+9))=(5+13)=18<eos>
<bos>((6+((8+5)+4))+2)=((6+(13+4))+2)=((6+17)+2)=(23+2)=25<eos>
<bos>(((3+7)+(6+2))+2)=((10+8)+2)=(18+2)=20<eos>
<bos>((((1+2)+7)+5)+2)=(((3+7)+5)+2)=((10+5)+2)=(15+2)=17<eos>
<bos>((7+2)+((5+7)+7))=(9+(12+7))=(9+19)=28<eos>
<bos>((5+7)+((8+1)+2))=(12+(9+2))=(12+11)=23<eos>
<bos>(((4+8)+(8+2))+5)=((12+10)+5)=(22+5)=27<eos>


## Implement a model that generalizes to more numbers and larger numbers


The sample code that follows is based on this ChatGPT session.

https://chatgpt.com/share/69036c83-171c-800c-9216-0884476017c6

In [11]:
def make_batch(n, *,digit_level=1, batch_size=64):
    if digit_level==1:
        value_min = 1
        value_max = 9
        
    elif digit_level==2:
        value_min = 10
        value_max = 99

    elif digit_level==3:
        value_min = 100
        value_max = 999

    else:
        raise ValueError(f"digit_level must be 1, 2, or 3, got {digit_level}") 

    seqs = [
        generate_instance(n, value_min=value_min, value_max=value_max)
        for _ in range(batch_size)
    ]
    

    # pad to max length on right
    batch = torch.nn.utils.rnn.pad_sequence(seqs, batch_first=True, padding_value=PAD)

    # next token targets: inputs are all but last; targets are all but first
    x = batch[:, :-1]
    y = batch[:, 1:]

    return x.to(device), y.to(device)

make_batch(5,digit_level=3)

(tensor([[ 0,  3,  3,  ...,  8,  8,  1],
         [ 0,  3,  3,  ..., 13, 10,  7],
         [ 0,  3,  3,  ...,  2,  2,  2],
         ...,
         [ 0,  3, 14,  ...,  2,  2,  2],
         [ 0,  3,  3,  ...,  2,  2,  2],
         [ 0,  3,  3,  ...,  2,  2,  2]], device='cuda:0'),
 tensor([[ 3,  3, 10,  ...,  8,  1,  2],
         [ 3,  3,  3,  ..., 10,  7,  1],
         [ 3,  3, 12,  ...,  2,  2,  2],
         ...,
         [ 3, 14,  9,  ...,  2,  2,  2],
         [ 3,  3,  3,  ...,  2,  2,  2],
         [ 3,  3,  3,  ...,  2,  2,  2]], device='cuda:0'))

In [12]:
def causal_mask(T):
    # shape (T, T); True = mask (disallow), False = keep
    # nn.Transformer expects float mask or bool depending on API;
    # TransformerEncoder uses src_mask where non-zero entries are masked.
    # We'll use a float mask with -inf above diagonal.
    m = torch.full((T, T), float("-inf"), device=device)
    m = torch.triu(m, diagonal=1)  # upper triangle is masked
    return m

In [13]:
# YOUR CHANGES HERE

class MathTransformer(torch.nn.Module):
    def __init__(self, d_model=128, nhead=4, num_layers=4, dim_ff=256, max_len=64, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.max_len = max_len

        vocab_size = len(TOKENS)

        # token + position embeddings
        self.tok_emb = torch.nn.Embedding(vocab_size, d_model, padding_idx=PAD)
        self.pos_emb = torch.nn.Embedding(max_len, d_model)

        layer = torch.nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_ff,
            dropout=dropout, batch_first=True,
        )
        self.blocks = torch.nn.TransformerEncoder(layer, num_layers=num_layers)
        self.lm_head = torch.nn.Linear(d_model, vocab_size)

        # init
        torch.nn.init.normal_(self.tok_emb.weight, mean=0.0, std=0.02)
        torch.nn.init.normal_(self.pos_emb.weight, mean=0.0, std=0.02)
        torch.nn.init.normal_(self.lm_head.weight, mean=0.0, std=0.02)
        torch.nn.init.zeros_(self.lm_head.bias)

    def forward(self, x):
        # x: (N, T)
        N, T = x.shape
        pos = torch.arange(T, device=x.device).unsqueeze(0)  # (1, T)
        h = self.tok_emb(x) * math.sqrt(self.d_model) + self.pos_emb(pos)  # (N, T, d_model)

        # key padding mask: True where we want to ignore (PAD)
        key_padding_mask = (x == PAD)  # (N, T) bool

        # causal mask for self-attention (float, -inf above diagonal)
        attn_mask = causal_mask(T) # (T, T)

        h = self.blocks(
            h,
            mask=attn_mask,                         # causal
            src_key_padding_mask=key_padding_mask   # pad masking
        )
        logits = self.lm_head(h)  # (N, T, vocab)
        return logits

    @torch.no_grad()
    def generate(self, prefix_ids, max_new_tokens=8):
        self.eval()
        x = prefix_ids.clone().to(next(self.parameters()).device)  # (N, T0)
        for _ in range(max_new_tokens):
            if x.size(1) >= self.max_len:
                break
            logits = self.forward(x)[:, -1, :]   # (N, V)
            next_id = torch.argmax(logits, dim=-1, keepdim=True)  # greedy
            x = torch.cat([x, next_id], dim=1)
            if (next_id == EOS).all():
                break
        return x

test_model = MathTransformer(d_model=128, nhead=4, num_layers=4, dim_ff=256, max_len=128, dropout=0.1)

In [47]:
def sample_phase(phase, batch_size=1024):
    if phase==1:
        digit_level= 1
        n = random.choice([2,3])

    elif phase==2:
        digit_level=random.choice([1,2])
        n = random.choice([2,3,4])

    elif phase==3:
        digit_level=random.choice([1,2,3])
        n = random.choice([2,3,4,5])

    else:
        raise ValueError("phase out of range")

    x, y = make_batch(
            n=n,
            digit_level= digit_level,
            batch_size=batch_size
    )
    return x,y

In [48]:
phase_and_steps = [
    (1, 3000),  # (phase, steps)
    (2, 6000),
    (3, 9000),
]

model = MathTransformer(
        d_model=192,
        nhead=6, 
        num_layers=6, 
        dim_ff=384,
        max_len=128, 
        dropout=0.1
).to(device)

criterion = torch.nn.CrossEntropyLoss(ignore_index=PAD)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4)

model.train()
global_steps = 0

for phase, steps in phase_and_steps:
    print(f"\n===== Start Phase {phase} for {steps} steps =====")
    model.train()
    
    for step in range(1, steps+1):
        x, y = sample_phase(phase,batch_size=1024)
        logits = model(x)                  # (N, T, V)
        loss = criterion(logits.reshape(-1, len(TOKENS)), y.reshape(-1))
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        global_steps+=1
        
        if step % 1000 == 0:
            print(f"step {step:5d} | global_steps {global_steps:5d} | loss {loss.item():.4f}")
    
    # print(f"\n[Phase {phase}] quick sanity check:")
    # for _ in range(5):
    #     test_example(model, n=3, verbose=True)


===== Start Phase 1 for 3000 steps =====


/share/pkg.8/academic-ml/fall-2025/install/fall-2025-pyt/lib/python3.12/site-packages/torch/nn/functional.py:6041: UserWarning: Support for mismatched src_key_padding_mask and mask is deprecated. Use same type for both instead.
  warnings.warn(


step  1000 | global_steps  1000 | loss 0.5710
step  2000 | global_steps  2000 | loss 0.5833
step  3000 | global_steps  3000 | loss 0.5946

===== Start Phase 2 for 6000 steps =====
step  1000 | global_steps  4000 | loss 1.5698
step  2000 | global_steps  5000 | loss 0.5559
step  3000 | global_steps  6000 | loss 0.5377
step  4000 | global_steps  7000 | loss 0.6911
step  5000 | global_steps  8000 | loss 0.4851
step  6000 | global_steps  9000 | loss 0.5720

===== Start Phase 3 for 9000 steps =====
step  1000 | global_steps 10000 | loss 0.8483
step  2000 | global_steps 11000 | loss 0.6942
step  3000 | global_steps 12000 | loss 0.4990
step  4000 | global_steps 13000 | loss 0.9551
step  5000 | global_steps 14000 | loss 0.7579
step  6000 | global_steps 15000 | loss 0.4873
step  7000 | global_steps 16000 | loss 0.6449
step  8000 | global_steps 17000 | loss 0.3784
step  9000 | global_steps 18000 | loss 0.4309


In [49]:
def prepare_prompt(s):
    token_ids = encode(s)
    if '=' in s:
        token_ids = token_ids[:s.index('=')+2]
        assert token_ids[-1] == TOKEN_MAP['=']

    return torch.tensor([token_ids], dtype=torch.long, device=device)

In [50]:
decode(generate_instance(3))

'<bos>((8+7)+5)=(15+5)=20<eos>'

In [51]:
def test_example(model,*args, verbose=True, **kwargs):
    model.eval()

    target_token_ids = generate_instance(*args, **kwargs)
    target = decode(target_token_ids)

    prompt = target[:target.index('=')+1]
    prompt_token_ids = encode(prompt, eos=False)
    prompt_batch = prompt_token_ids.reshape(shape=(1,-1))

    actual_token_ids = model.generate(prompt_batch, max_new_tokens=50)[0]
    actual = decode(actual_token_ids)

    correct = actual == target

    if verbose or not correct:
        print("PROMPT", decode(prompt_token_ids), "TARGET", target, "ACTUAL", actual, "CORRECT", correct)

    return correct

test_example(model,n=3)

PROMPT <bos>((9+4)+8)= TARGET <bos>((9+4)+8)=(13+8)=21<eos> ACTUAL <bos>((9+4)+8)=(13+8)=21<eos> CORRECT True


True

In [52]:
for _ in range(10):
    test_example(model,n=3, verbose=True)

PROMPT <bos>(6+(9+8))= TARGET <bos>(6+(9+8))=(6+17)=23<eos> ACTUAL <bos>(6+(9+8))=(6+17)=23<eos> CORRECT True
PROMPT <bos>((5+4)+8)= TARGET <bos>((5+4)+8)=(9+8)=17<eos> ACTUAL <bos>((5+4)+8)=(9+8)=17<eos> CORRECT True
PROMPT <bos>((3+6)+8)= TARGET <bos>((3+6)+8)=(9+8)=17<eos> ACTUAL <bos>((3+6)+8)=(9+8)=17<eos> CORRECT True
PROMPT <bos>((2+7)+6)= TARGET <bos>((2+7)+6)=(9+6)=15<eos> ACTUAL <bos>((2+7)+6)=(9+6)=15<eos> CORRECT True
PROMPT <bos>(5+(4+3))= TARGET <bos>(5+(4+3))=(5+7)=12<eos> ACTUAL <bos>(5+(4+3))=(5+7)=12<eos> CORRECT True
PROMPT <bos>(1+(3+6))= TARGET <bos>(1+(3+6))=(1+9)=10<eos> ACTUAL <bos>(1+(3+6))=(1+9)=10<eos> CORRECT True
PROMPT <bos>(5+(4+7))= TARGET <bos>(5+(4+7))=(5+11)=16<eos> ACTUAL <bos>(5+(4+7))=(5+11)=16<eos> CORRECT True
PROMPT <bos>((5+3)+3)= TARGET <bos>((5+3)+3)=(8+3)=11<eos> ACTUAL <bos>((5+3)+3)=(8+3)=11<eos> CORRECT True
PROMPT <bos>(7+(6+4))= TARGET <bos>(7+(6+4))=(7+10)=17<eos> ACTUAL <bos>(7+(6+4))=(7+10)=17<eos> CORRECT True
PROMPT <bos>((7+8)+5)=

In [53]:
for _ in range(10):
    test_example(model, n=4, verbose=True)

PROMPT <bos>(8+((2+1)+7))= TARGET <bos>(8+((2+1)+7))=(8+(3+7))=(8+10)=18<eos> ACTUAL <bos>(8+((2+1)+7))=(8+(3+7))=(8+10)=18<eos> CORRECT True
PROMPT <bos>((8+6)+(9+9))= TARGET <bos>((8+6)+(9+9))=(14+18)=32<eos> ACTUAL <bos>((8+6)+(9+9))=(14+18)=32<eos> CORRECT True
PROMPT <bos>(6+(1+(4+9)))= TARGET <bos>(6+(1+(4+9)))=(6+(1+13))=(6+14)=20<eos> ACTUAL <bos>(6+(1+(4+9)))=(6+(1+13))=(6+14)=20<eos> CORRECT True
PROMPT <bos>((7+2)+(9+1))= TARGET <bos>((7+2)+(9+1))=(9+10)=19<eos> ACTUAL <bos>((7+2)+(9+1))=(9+10)=19<eos> CORRECT True
PROMPT <bos>((9+8)+(7+6))= TARGET <bos>((9+8)+(7+6))=(17+13)=30<eos> ACTUAL <bos>((9+8)+(7+6))=(17+13)=30<eos> CORRECT True
PROMPT <bos>(7+(6+(3+8)))= TARGET <bos>(7+(6+(3+8)))=(7+(6+11))=(7+17)=24<eos> ACTUAL <bos>(7+(6+(3+8)))=(7+(6+11))=(7+17)=24<eos> CORRECT True
PROMPT <bos>((3+(7+6))+8)= TARGET <bos>((3+(7+6))+8)=((3+13)+8)=(16+8)=24<eos> ACTUAL <bos>((3+(7+6))+8)=((3+13)+8)=(16+8)=24<eos> CORRECT True
PROMPT <bos>((3+1)+(3+3))= TARGET <bos>((3+1)+(3+3))=(4+

### Benchmark your model

Test your code with different numbers of integers and numbers of input digits.
The `generate_instance` function provided uses the parameter `n` to control the number of integers, and `value_min` and `value_max` to control the range of integers.
For example, 2 input digits would correspond to `value_min=10` and `value_max=99`.

Test the accuracy on the combinations specified in the table below, and fill in your accuracy numbers in that table.
Make sure that you run enough samples for statistical significance (usually at least 1000 recommended) as your benchmarking accuracy will be checked for consistency with tests by the auto-grader.

In [54]:
# YOUR CHANGES HERE
n_values = list(range(2,6))
runs = 1000

for n in n_values:
    values_min = [1, 10, 100]
    values_max = [9, 99, 999]
    
    
    if n <= 3:
        max_i = 3
    else:
        max_i = 2
        
    for i in range(max_i):
        correct = 0
        
        for _ in range(runs):
            is_correct = test_example(
                model,
                n=n,
                value_min=values_min[i],
                value_max=values_max[i],
                verbose=False        # False -> only print correct false outcome
            )

            if is_correct:
                correct += 1

        accuracy = correct / runs
        print(f"n = {n} | input digits = {i+1} | acc = {accuracy:.4f}")

n = 2 | input digits = 1 | acc = 1.0000
n = 2 | input digits = 2 | acc = 1.0000
n = 2 | input digits = 3 | acc = 1.0000
n = 3 | input digits = 1 | acc = 1.0000
n = 3 | input digits = 2 | acc = 1.0000
n = 3 | input digits = 3 | acc = 1.0000
n = 4 | input digits = 1 | acc = 1.0000
n = 4 | input digits = 2 | acc = 1.0000
n = 5 | input digits = 1 | acc = 1.0000
n = 5 | input digits = 2 | acc = 1.0000


Fill in this table.

| n | input digits | accuracy |
|---|---|-----|
| 2 | 1 | 1.0000 |
| 2 | 2 | 1.0000 |
| 2 | 3 | 1.0000 |
| 3 | 1 | 1.0000 |
| 3 | 2 | 1.0000 |
| 3 | 3 | 1.0000 |
| 4 | 1 | 1.0000 |
| 4 | 2 | 1.0000 |
| 5 | 1 | 1.0000 |
| 5 | 2 | 1.0000 |

Do not change the table header as the auto-grader will use it to check your results.


## Save model and implement a command line interface.

Your model will be tested automatically with a suite of examples with different numbers of values and digits matching your previous benchmark task.
For this testing, you must save your model weights and write a program to run your model.

### Save your model weights.

Save your model weights as `math.pt` to be submitted in Gradescope.

In [22]:
# YOUR CHANGES HERE
torch.save(model.state_dict(), "math.pt")

### Write a program to run your model.

Write a Python script `predict.py` that takes a single filename as input, reads each line as a prompt, generates the completion, and writes out the result to standard output.
We will invoke your program with a command like `python3 predict.py INPUT.txt` and capture the standard output for grading.

The input file will not include the special tokens such as `<bos>` or `<eos>`.
Similarly, your output should not include them either.

For example, given an input file with the following contents,
```
(((1+2)+1)+8)=
```
your program should write the following output.
```
(((1+2)+1)+8)=((3+1)+8)=(4+8)=12
```


In [1]:
%%writefile predict.py
import sys
import math
import torch


device = torch.device("cpu")


characters = "()+0123456789="
TOKENS = ["<bos>", "<eos>", "<pad>"] + [c for c in characters]

TOKEN_MAP = dict((t, i) for i, t in enumerate(TOKENS))

BOS = TOKEN_MAP["<bos>"]
EOS = TOKEN_MAP["<eos>"]
PAD = TOKEN_MAP["<pad>"]


def encode(s, *, eos = False):
    """
    transformed input string into token id：
    [BOS, <chars...>, (optional EOS)]
    """
    if s.startswith("<bos>"):
        s = s[5:]

    output = [BOS]

    for c in s:
        if c ==" ":
            continue
            
        output.append(TOKEN_MAP[c])

    if eos:
        output.append(EOS)

    return torch.tensor(output, device=device, dtype=torch.long)


def decode(token_ids):
    """
    turn token_id back into string and remove <bos> <eos> <pad>
    """
    chars = []
    for idx in token_ids:
        tok = TOKENS[int(idx)]
        if tok in ("<bos>", "<eos>", "<pad>"):
            continue
        chars.append(tok)
    return "".join(chars)


def causal_mask(T):
    m = torch.full((T, T), float("-inf"))
    m = torch.triu(m, diagonal=1)
    return m



class MathTransformer(torch.nn.Module):
    def __init__(self, d_model=192, nhead=6, num_layers=6, dim_ff=384, max_len=128, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.max_len = max_len

        vocab_size = len(TOKENS)

        # token + position embeddings
        self.tok_emb = torch.nn.Embedding(vocab_size, d_model, padding_idx=PAD)
        self.pos_emb = torch.nn.Embedding(max_len, d_model)

        layer = torch.nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_ff,
            dropout=dropout,
            batch_first=True,
        )
        self.blocks = torch.nn.TransformerEncoder(layer, num_layers=num_layers)
        self.lm_head = torch.nn.Linear(d_model, vocab_size)

        # init
        torch.nn.init.normal_(self.tok_emb.weight, mean=0.0, std=0.02)
        torch.nn.init.normal_(self.pos_emb.weight, mean=0.0, std=0.02)
        torch.nn.init.normal_(self.lm_head.weight, mean=0.0, std=0.02)
        torch.nn.init.zeros_(self.lm_head.bias)

    def forward(self, x):
        # x: (N, T)
        N, T = x.shape
        pos = torch.arange(T, device=x.device).unsqueeze(0)  # (1, T)
        h = self.tok_emb(x) * math.sqrt(self.d_model) + self.pos_emb(pos)  # (N, T, d_model)

        # key padding mask: True where we want to ignore (PAD)
        key_padding_mask = (x == PAD)  # (N, T) bool

        # causal mask for self-attention (float, -inf above diagonal)
        attn_mask = causal_mask(T).to(x.device)  # (T, T)

        h = self.blocks(
            h,
            mask=attn_mask,                        # causal
            src_key_padding_mask=key_padding_mask  # pad masking
        )
        logits = self.lm_head(h)  # (N, T, vocab)
        return logits

    @torch.no_grad()
    def generate(self, prefix_ids, max_new_tokens=128):
        self.eval()
        x = prefix_ids.clone().to(next(self.parameters()).device)  # (N, T0)
        for _ in range(max_new_tokens):
            if x.size(1) >= self.max_len:
                break
            logits = self.forward(x)[:, -1, :]   # (N, V)
            next_id = torch.argmax(logits, dim=-1, keepdim=True)  # greedy: (N, 1)
            x = torch.cat([x, next_id], dim=1)
            if (next_id == EOS).all():
                break
        return x



# --------------------------------------------------
def main():
    if len(sys.argv) != 2:
        sys.exit(1)

    input_filename = sys.argv[1]

    # Model building 
    model = MathTransformer(
        d_model=192,
        nhead=6,
        num_layers=6,
        dim_ff=384,
        max_len=128,
        dropout=0.1,
    ).to(device)

    state_dict = torch.load("math.pt", map_location=device)
    model.load_state_dict(state_dict)
    model.eval()

    
    with open(input_filename, "r") as f:
        for line in f:
            prompt = line.strip()
            if prompt == "":
                print("")
                continue


            prefix = encode(prompt, eos=False).unsqueeze(0)  # (1, T)

            with torch.no_grad():
                full_ids = model.generate(prefix, max_new_tokens=128)[0].tolist()

            output_str = decode(full_ids)
            print(output_str)


if __name__ == "__main__":
    main()

Overwriting predict.py


In [2]:
%%writefile test.py
import sys
import math
import torch
import random

characters = "()+0123456789="
TOKEN_MAP = dict((t, i) for i, t in enumerate(characters))

def input_generate(n, digits_level=1):
    nums = []

    for _ in range(n):
        if digits_level==1:
            num_generation = random.randint(0,9)
        else:
            low = 10 **(digits_level-1)
            high = 10**digits_level -1
            num_generation = random.randint(low,high)
        nums.append(str(num_generation))

    expression_list = nums.copy()

    while len(expression_list)>1:
        i = random.randint(0, len(expression_list) -2)
        merged = f"({expression_list[i]} + {expression_list[i+1]})"
        expression_list = expression_list[:i] + [merged] + expression_list[i+2:]
        
    return expression_list[0]

def main():
    num_lines = 30
    output_file = 'input_test.txt'

    with open(output_file, "w") as f:
        for _ in range(num_lines):
            num = random.randint(2,5)
            if num <=3:
                digits = random.choice([1,2,3])
            else:
                digits = random.choice([1,2])
            expression = input_generate(n = num, digits_level=digits)
            f.write(expression + "\n")
    print(f"Randomly generate {num_lines} tested expressions in {output_file}")

if __name__ == "__main__":
    main()

Overwriting test.py


In [84]:
!echo "(((1+2)+1)+8)=" > INPUT.txt
!python3 predict.py INPUT.txt

/share/pkg.8/academic-ml/fall-2025/install/fall-2025-pyt/lib/python3.12/site-packages/torch/nn/functional.py:6041: UserWarning: Support for mismatched src_key_padding_mask and mask is deprecated. Use same type for both instead.
  warnings.warn(
(((1+2)+1)+8)=((3+1)+8)=(4+8)=12


In [3]:
!python3 test.py
!python3 predict.py input_test.txt

Randomly generate 30 tested expressions in input_test.txt
/share/pkg.8/academic-ml/fall-2025/install/fall-2025-pyt/lib/python3.12/site-packages/torch/nn/functional.py:6041: UserWarning: Support for mismatched src_key_padding_mask and mask is deprecated. Use same type for both instead.
  warnings.warn(
(32+((96+63)+34))=(32+(159+34))=(32+193)=225
(4+2)=6
((44+(93+40))+(52+79))=((44+133)+131)=(177+131)=308
(236+175)=411
((9+0)+6)=(9+6)=15
(390+362)=752
(2+(5+((6+8)+3)))=(2+(5+(14+3)))=(2+(5+17))=(2+22)=24
((47+29)+(97+38))=(76+135)=211
((6+((5+2)+9))+0)=((6+(7+9))+0)=((6+16)+0)=(22+0)=22
(22+(77+(64+59)))=(22+(77+123))=(22+200)=222
(3+(1+(9+4)))=(3+(1+13))=(3+14)=17
(557+572)=1129
((15+(10+30))+(31+99))=((15+40)+130)=(55+130)=185
(485+(844+888))=(485+1732)=2217
((7+(8+9))+1)=((7+17)+1)=(24+1)=25
(0+((2+1)+(5+4)))=(0+(3+9))=(0+12)=12
((55+(57+41))+26)=((55+98)+26)=(153+26)=179
(((8+7)+2)+(2+5))=((15+2)+7)=(17+7)=24
(0+((5+2)+7))=(0+(7+7))=(0+14)=14
(((8+2)+(1+0))+5)=((10+11)+5)=(21+5)=26


## Final Submission

Submit your copy of this notebook with all your code, your saved model "best.py", and your prediction script "predict.py" to Gradescope.
